In [1]:
from ucimlrepo import fetch_ucirepo 
import numpy as np
import seaborn as sb
import pandas as pd
  
# fetch dataset 
congressional_voting_records = fetch_ucirepo(id=105) 
  
# data (as pandas dataframes) 
X = congressional_voting_records.data.features 
y = congressional_voting_records.data.targets 



In [2]:
X.shape

(435, 16)

In [3]:
X.isnull().sum()

handicapped-infants                        12
water-project-cost-sharing                 48
adoption-of-the-budget-resolution          11
physician-fee-freeze                       11
el-salvador-aid                            15
religious-groups-in-schools                11
anti-satellite-test-ban                    14
aid-to-nicaraguan-contras                  15
mx-missile                                 22
immigration                                 7
synfuels-corporation-cutback               21
education-spending                         31
superfund-right-to-sue                     25
crime                                      17
duty-free-exports                          28
export-administration-act-south-africa    104
dtype: int64

In [4]:
X.head()

,handicapped-infants,water-project-cost-sharing,adoption-of-the-budget-resolution,physician-fee-freeze,el-salvador-aid,religious-groups-in-schools,anti-satellite-test-ban,aid-to-nicaraguan-contras,mx-missile,immigration,synfuels-corporation-cutback,education-spending,superfund-right-to-sue,crime,duty-free-exports,export-administration-act-south-africa
0,n,y,n,y,y,y,n,n,n,y,NaN,y,y,y,n,y
1,n,y,n,y,y,y,n,n,n,n,n,y,y,y,n,NaN
2,NaN,y,y,NaN,y,y,n,n,n,n,y,n,y,y,n,n
3,n,y,y,n,NaN,y,n,n,n,n,y,n,y,n,n,y
4,y,y,y,n,y,y,n,n,n,n,y,NaN,y,y,y,y


In [5]:
# 2. Rimuovi "physician-fee-freeze"
X = X.drop(columns=['physician-fee-freeze'])

# 3. Unisci
df = pd.concat([X, y], axis=1)

# 4. Imputazione per partito (in 3 righe!)
for col in df.columns[:-1]:  # per ogni feature
    # Calcola la moda per partito e usala per riempire i NaN
    df[col] = df.groupby('Class')[col].transform(
        lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'n')
    )

# 5. Converti in numeri
df = df.replace({'y': 1, 'n': 0})

# 6. Separa
X_clean = df.drop(columns=['Class'])
y_clean = df['Class']
y_clean = df['Class'].map({'democrat': 0, 'republican': 1}).values.astype(np.int64)

C:\Users\david\AppData\Local\Temp\ipykernel_26092\2161300700.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace({'y': 1, 'n': 0})


In [6]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [7]:
# Dividi in train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y_clean, test_size=0.2, random_state=42
)

In [8]:
# Addestra una rete (come nel paper)
mlp = MLPClassifier(
    hidden_layer_sizes=(10,),  # prova 0,5,10,20,40
    max_iter=1000,
    random_state=42
)
mlp.fit(X_train, y_train)
y_pred = mlp.predict(X_test)

In [9]:
print('Test Accuracy %s' % accuracy_score(y_test, y_pred))
print('Test F1-score %s' % f1_score(y_test, y_pred, average=None))
print(classification_report(y_test, y_pred))

Test Accuracy 0.9310344827586207
Test F1-score [0.94827586 0.89655172]
              precision    recall  f1-score   support

           0       0.92      0.98      0.95        56
           1       0.96      0.84      0.90        31

    accuracy                           0.93        87
   macro avg       0.94      0.91      0.92        87
weighted avg       0.93      0.93      0.93        87



In [10]:
# importa la classe direttamente dal file
from RuleTree.tree.TrepanClassifier import TrepanClassifier
trepan_clf = TrepanClassifier(estimator = mlp, s_min=1000, random_state=42, max_leaf_nodes=16)
trepan_clf.fit(X_train, y_train)
y_pred_trepan = trepan_clf.predict(X_test.values)
print('Accuracy %s' % accuracy_score(y_test, y_pred_trepan))
print('F1-score %s' % f1_score(y_test, y_pred_trepan, average=None))
print(classification_report(y_test, y_pred_trepan))
fidelty = accuracy_score(y_pred, y_pred_trepan)
print("Fidelity of Trepan to the original model:", fidelty)




c:\Users\david\miniconda3\envs\trepan-dev\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accuracy 0.9310344827586207
F1-score [0.94827586 0.89655172]
              precision    recall  f1-score   support

           0       0.92      0.98      0.95        56
           1       0.96      0.84      0.90        31

    accuracy                           0.93        87
   macro avg       0.94      0.91      0.92        87
weighted avg       0.93      0.93      0.93        87

Fidelity of Trepan to the original model: 1.0


In [11]:
def create_mlp(hidden_units):
    if hidden_units == 0:
        return MLPClassifier(
            hidden_layer_sizes=(),
            max_iter=1000,  # aumentato
            random_state=42,
            early_stopping=False,
        )
    else:
        return MLPClassifier(
            hidden_layer_sizes=(hidden_units,),
            max_iter=1000,
            random_state=42,
            early_stopping=False,
        )

def select_best_hidden_units(X_train, y_train, hidden_units_list, cv_inner=5):
    """
    Seleziona il miglior numero di hidden unit con cross-validation interna.
    Come nel paper: prova {0,5,10,20,40} e sceglie il migliore.
    """
    X_train = np.asarray(X_train, dtype=np.float64)
    # y_train è già int, non convertire
    
    best_units = 10
    best_score = -1
    
    for units in hidden_units_list:
        mlp = create_mlp(units)
        scores = cross_val_score(mlp, X_train, y_train, cv=cv_inner, scoring='accuracy')
        mean_score = np.mean(scores)
        
        if mean_score > best_score:
            best_score = mean_score
            best_units = units
            
    return best_units, best_score

def train_and_evaluate_network(X_train, y_train, X_test, y_test, hidden_units):
    """
    Addestra una rete neurale e restituisce:
    - accuracy sul test set
    - modello addestrato
    - predizioni sul test set
    """
    mlp = create_mlp(hidden_units)
    mlp.fit(X_train, y_train)
    y_pred = mlp.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return acc, mlp, y_pred

In [12]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [13]:
hidden_units_list =  [0,5,10, 20, 40] # come nel paper
n_folds = 10
random_state = 42

# Inizializza metriche
accuracies_net = []
accuracies_tree = []
fidelities = []
chosen_units = []

skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)

print("\n" + "="*60)
print("INIZIO 10-FOLD CROSS-VALIDATION")
print("="*60 + "\n")

# ============================================================
# 6. 10-FOLD CROSS-VALIDATION
# ============================================================
for fold, (train_idx, test_idx) in enumerate(skf.split(X_clean, y_clean), 1):
    print(f"Fold {fold}/{n_folds}")
    
    X_train = X_clean.iloc[train_idx]  # se X_clean è DataFrame
    y_train = y_clean[train_idx]       # se y_clean è Series/array
    X_test = X_clean.iloc[test_idx]
    y_test = y_clean[test_idx]

    # Seleziona il miglior numero di hidden unit
    best_units, best_cv_score = select_best_hidden_units(
        X_train, y_train, hidden_units_list, cv_inner=5
    )
    chosen_units.append(best_units)
    print(f"  → Migliori hidden units: {best_units} (CV score interno: {best_cv_score:.3f})")
    
    # Addestra la rete finale
    net_acc, network, y_pred_net = train_and_evaluate_network(
        X_train, y_train, X_test, y_test, best_units
    )
    accuracies_net.append(net_acc)
    print(f"  → Accuratezza rete: {net_acc:.3f}")
    
    # ============================================================
    # 7. TREPAN - Estrazione dell'albero
    # ============================================================
    
    # Se TREPAN è importato, usalo
    trepan_clf = TrepanClassifier(estimator=network, s_min=1000, random_state=42, max_leaf_nodes=16)
    trepan_clf.fit(X_train, y_train)
    y_pred_trepan = trepan_clf.predict(X_test.values)
    
    
    
    tree_acc = accuracy_score(y_test, y_pred_trepan)
    fidelity = accuracy_score(y_pred_net, y_pred_trepan)
    
    accuracies_tree.append(tree_acc)
    fidelities.append(fidelity)
    
    print(f"  → Accuratezza albero: {tree_acc:.3f}")
    print(f"  → Fedeltà albero-rete: {fidelity:.3f}")
    print()

# ============================================================
# 8. RISULTATI FINALI
# ============================================================
print("="*60)
print("RISULTATI FINALI (10-fold CV)")
print("="*60)
print(f"Accuratezza rete:     {np.mean(accuracies_net):.3f} ± {np.std(accuracies_net):.3f}")
print(f"Accuratezza albero:   {np.mean(accuracies_tree):.3f} ± {np.std(accuracies_tree):.3f}")
print(f"Fedeltà albero-rete:  {np.mean(fidelities):.3f} ± {np.std(fidelities):.3f}")
print(f"Hidden units più scelte: {pd.Series(chosen_units).value_counts().to_dict()}")


INIZIO 10-FOLD CROSS-VALIDATION

Fold 1/10
  → Migliori hidden units: 5 (CV score interno: 0.941)
  → Accuratezza rete: 0.909
  → Accuratezza albero: 0.955
  → Fedeltà albero-rete: 0.955

Fold 2/10
  → Migliori hidden units: 10 (CV score interno: 0.928)
  → Accuratezza rete: 0.977
  → Accuratezza albero: 0.955
  → Fedeltà albero-rete: 0.977

Fold 3/10
  → Migliori hidden units: 5 (CV score interno: 0.928)
  → Accuratezza rete: 0.955
  → Accuratezza albero: 0.977
  → Fedeltà albero-rete: 0.977

Fold 4/10
  → Migliori hidden units: 10 (CV score interno: 0.936)
  → Accuratezza rete: 0.864
  → Accuratezza albero: 0.818
  → Fedeltà albero-rete: 0.909

Fold 5/10
  → Migliori hidden units: 5 (CV score interno: 0.933)
  → Accuratezza rete: 0.955
  → Accuratezza albero: 0.955
  → Fedeltà albero-rete: 1.000

Fold 6/10
  → Migliori hidden units: 10 (CV score interno: 0.931)
  → Accuratezza rete: 0.860
  → Accuratezza albero: 0.860
  → Fedeltà albero-rete: 1.000

Fold 7/10
  → Migliori hidden uni